## COOTEFOO – Data Understanding

Esplorazione sui tre knowledge graph forniti:

- **FILAH.json** – dataset raccolto/usato da *Fishing Is Living And Heritage*
- **TROUT.json** – dataset raccolto/usato da *Tourism Raises OceanUs Together*
- **journalist.json** – dataset "completo" acquisito dalla giornalista (FILAH + TROUT + record aggiuntivi)

più due file geografici di supporto:

- **oceanus_map.geojson** – confini/zone dell'arcipelago (isole, aree)
- **road_map.json** – rete stradale (nodi/archi), utile per calcoli di distanza/tempo di viaggio

Obiettivo del notebook:

1. capire la struttura dei knowledge graph (tipi di nodo/arco, schema, campi disponibili);
2. quantificare **cosa manca** in FILAH e TROUT rispetto al dataset completo (copertura di membri, meeting, topic);
3. individuare pattern grezzi di **bias** (sentiment per persona/industria, frequenza dei topic) che poi guideranno il disegno delle viz per i 4 task;
4. capire se/come i dati geografici (place, trip, road network) possano essere usati.

> Nota: i dataset sono in formato *node-link* (stile NetworkX `node_link_data`), multigrafo diretto.


In [1]:
import json
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

FILES = {
    "FILAH": "../public/data/raw_data/FILAH.json",
    "TROUT":  "../public/data/raw_data/TROUT.json",
    "journalist": "../public/data/raw_data/journalist.json",
}


/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alicecalderini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


## 1. Caricamento dei grafi


In [2]:
def load_graph(path):
    with open(path) as f:
        data = json.load(f)
    G = nx.node_link_graph(data, link="links")
    return G

graphs = {name: load_graph(p) for name, p in FILES.items()}

for name, G in graphs.items():
    print(f"{name:12s}  nodes={G.number_of_nodes():4d}   edges={G.number_of_edges():4d}   "
          f"directed={G.is_directed()}  multigraph={G.is_multigraph()}")


FILAH         nodes= 396   edges= 765   directed=True  multigraph=True
TROUT         nodes= 164   edges= 378   directed=True  multigraph=True
journalist    nodes= 740   edges=2436   directed=True  multigraph=True


In [3]:
graphs

{'FILAH': <networkx.classes.multidigraph.MultiDiGraph at 0x7fd0eed1b880>,
 'TROUT': <networkx.classes.multidigraph.MultiDiGraph at 0x7fd0eed1bc70>,
 'journalist': <networkx.classes.multidigraph.MultiDiGraph at 0x7fd0f3bf9610>}

## RICERCA MISSING VALUES (NODI SENZA TYPE)

type esistenti:

In [4]:
type_set = set()
for node, attr in graphs["journalist"].nodes(data=True):
        #print(node, attr) #node è il nome del nodo
        attrtype = attr.get("type", "NA")
        type_set.add(attrtype)
type_set

{'NA',
 'discussion',
 'entity.organization',
 'entity.person',
 'meeting',
 'place',
 'plan',
 'topic',
 'trip'}

In [5]:
# --- conteggio nodi per tipo, per dataset ---
node_types = ['entity.person', 'entity.organization', 'meeting',
              'discussion', 'plan', 'topic', 'place', 'trip']

table = {}
for name, G in graphs.items():
    counts = Counter(attrs.get('type') for _, attrs in G.nodes(data=True))
    table[name] = {t: counts.get(t, 0) for t in node_types}

header = f"{'Nodo':22s}" + ''.join(f"{n:>12s}" for n in FILES)
print(header)
for t in node_types:
    row = f"{t:22s}" + ''.join(f"{table[name][t]:12d}" for name in FILES)
    print(row)

Nodo                         FILAH       TROUT  journalist
entity.person                    3           6           6
entity.organization              8           8           8
meeting                         12          13          16
discussion                      58          39         101
plan                            41          33          74
topic                           14          14          15
place                           59          30         159
trip                           189          18         342


In [6]:
journalist_NA_nodes = set()
for node, attr in graphs["journalist"].nodes(data=True):
        #print(node, attr) #node è il nome del nodo
        attrtype = attr.get("type", "NA")
        if attrtype == "NA":
                print(node, attr)
                journalist_NA_nodes.add(node)

journalist_NA_nodes

35889363 {'lat': -165.68041773674673, 'lon': 38.98946585258907, 'zone': 'industrial', 'zone_detail': None, 'label': 'Harbor Route Solutions'}
3753651731 {'lat': -165.88631162829017, 'lon': 39.65407871983316, 'zone': 'industrial', 'zone_detail': None, 'label': 'High Seas Fishing Inc.'}
491334376 {'lat': -165.59436715313234, 'lon': 39.55699548649617, 'zone': 'industrial', 'zone_detail': None, 'label': 'Blue Wave Shipping'}
Bay Harvest Corporation {}
30192735 {'lat': -165.88484202581887, 'lon': 39.65638234598041, 'zone': 'industrial', 'zone_detail': None, 'label': 'Bay Harvest Corporation'}
35922181 {'lat': -165.67106823674675, 'lon': 38.99472785258907, 'zone': 'industrial', 'zone_detail': None, 'label': 'Harbor Builders'}
35872967 {'lat': -165.67991093674675, 'lon': 38.99139455258907, 'zone': 'industrial', 'zone_detail': None, 'label': 'Haacklee Assembly Co.'}
48468290 {'lat': -165.60067726355345, 'lon': 39.52500051834688, 'zone': 'residential', 'zone_detail': None, 'label': 'Waveside To

{10803677425,
 30192735,
 35844638,
 35872967,
 35889363,
 35908634,
 35913119,
 35922181,
 3753651731,
 37831107,
 48468290,
 48480127,
 491334376,
 5849775342,
 'Bay Harvest Corporation',
 'Harbor Odyssey Tours',
 'Sean',
 'concert_Travel_Harborfront_Market',
 'name_harbor_area_Meeting_11_Harbor_Odyssey_Tours'}

In [7]:
FILAH_NA_nodes = set()
for node, attr in graphs["FILAH"].nodes(data=True):
        #print(node, attr) #node è il nome del nodo
        attrtype = attr.get("type", "NA")
        if attrtype == "NA":
                print(node, attr)
                FILAH_NA_nodes.add(node)
FILAH_NA_nodes

35889363 {'lat': -165.68041773674673, 'lon': 38.98946585258907, 'zone': 'industrial', 'zone_detail': None, 'label': 'Harbor Route Solutions'}
3753651731 {'lat': -165.88631162829017, 'lon': 39.65407871983316, 'zone': 'industrial', 'zone_detail': None, 'label': 'High Seas Fishing Inc.'}
491334376 {'lat': -165.59436715313234, 'lon': 39.55699548649617, 'zone': 'industrial', 'zone_detail': None, 'label': 'Blue Wave Shipping'}
30192735 {'lat': -165.88484202581887, 'lon': 39.65638234598041, 'zone': 'industrial', 'zone_detail': None, 'label': 'Bay Harvest Corporation'}
35922181 {'lat': -165.67106823674675, 'lon': 38.99472785258907, 'zone': 'industrial', 'zone_detail': None, 'label': 'Harbor Builders'}
35872967 {'lat': -165.67991093674675, 'lon': 38.99139455258907, 'zone': 'industrial', 'zone_detail': None, 'label': 'Haacklee Assembly Co.'}
48468290 {'lat': -165.60067726355345, 'lon': 39.52500051834688, 'zone': 'residential', 'zone_detail': None, 'label': 'Waveside Townhomes'}
9196630064 {'lat'

{30192735,
 35844638,
 35872967,
 35889363,
 35908634,
 35922181,
 48468290,
 48475647,
 491334376,
 581717431,
 3753651731,
 9196630064}

In [8]:
TROUT_NA_nodes = set()
for node, attr in graphs["TROUT"].nodes(data=True):
        #print(node, attr) #node è il nome del nodo
        attrtype = attr.get("type", "NA")
        if attrtype == "NA":
                #print(node, attr)
                TROUT_NA_nodes.add(node)
TROUT_NA_nodes

{35889363, 48468290, 48480127}

In [9]:
print(TROUT_NA_nodes.issubset(journalist_NA_nodes))

True


In [10]:
print(FILAH_NA_nodes.issubset(journalist_NA_nodes))

False


inoltre FILAH ha dei luoghi che in journalist invece ci sono

In [11]:
FILAH_difference = FILAH_NA_nodes - journalist_NA_nodes
FILAH_difference

{48475647, 581717431, 9196630064}

In [12]:
for node in FILAH_difference:
    print(graphs["journalist"].nodes[node])

{'lat': -165.88410403403049, 'lon': 39.66355038525348, 'zone': 'government', 'zone_detail': 'city hall', 'label': 'Himark City Hall', 'type': 'place', 'name': 'Himark City Hall'}
{'lat': -164.56100199599447, 'lon': 39.27513421446645, 'zone': 'government', 'zone_detail': 'environmental', 'label': 'Tropics Environmental Hub', 'type': 'place', 'name': 'Tropics Environmental Hub'}
{'lat': -165.59719356711753, 'lon': 39.521992579121736, 'zone': 'government', 'zone_detail': 'city hall', 'label': 'Lomark Civic Plaza', 'type': 'place', 'name': 'Lomark Civic Plaza'}


 i valori senza type sono delle zone e delle persone di cui non si sa nulla, ma sono solo nel dataset completo

## RICERCA MISSING VALUES (ARCHI SENZA ROLE)

In [13]:
role_set = set()
for _, _, dati in graphs["journalist"].edges(data=True):
        #print(node, attr) #node è il nome del nodo
        edgerole = dati.get("role", "trip-link")
        role_set.add(edgerole)
print(role_set)

{'trip-link', 'participant', 'travel', 'refers_to', 'part_of', 'plan', 'about'}


In [14]:
for u, v, dati in graphs["journalist"].edges(data=True):
        #print(node, attr) #node è il nome del nodo
        edgerole = dati.get("role", "trip-link")
        if edgerole == "participant":
                print(f" source {u}, target {v}, {dati}")


 source expanding_tourist_wharf_Meeting_7_Initial_Views_Discussion, target Tours Central Ticketing, {'role': 'participant', 'sentiment': 1, 'reason': 'More tourism so more tickets sold.', 'industry': ['tourism']}
 source expanding_tourist_wharf_Meeting_7_Initial_Views_Discussion, target Paackland Container Inc., {'role': 'participant', 'sentiment': -0.5, 'reason': 'Tourism is a competitor for the same resources.', 'industry': ['tourism']}
 source expanding_tourist_wharf_Meeting_7_Initial_Views_Discussion, target Seal, {'role': 'participant', 'sentiment': 0.1, 'reason': 'Seems like headache, but if other people are willing to do the work.', 'industry': ['tourism']}
 source expanding_tourist_wharf_Travel_Harbor_Route_Solutions_Discussion_Tourist, target Simone Kat, {'role': 'participant', 'sentiment': 0.5, 'reason': 'Supports tourism growth', 'industry': ['tourism']}
 source expanding_tourist_wharf_Travel_Harbor_Route_Solutions, target Simone Kat, {'role': 'participant', 'sentiment': 0.5

### è veramente trip-link? sì

In [15]:
def get_trip_links(graph):
    edges = []

    for u, v, dati in graph.edges(data=True):
        edgerole = dati.get("role", "trip-link")

        if edgerole == "trip-link":
            edge = {
                "u": u,
                "v": v,
                **dati
            }

            edges.append(edge)
            #print(u, v, dati)

    return edges


In [16]:

journalist_role_set = get_trip_links(graphs["journalist"])
trout_role_set = get_trip_links(graphs["TROUT"])
filah_role_set = get_trip_links(graphs["FILAH"])

In [17]:
print(journalist_role_set)

[{'u': 'trip_0', 'v': 'Simone Kat'}, {'u': 'trip_0', 'v': 'Haacklee Ferry Terminal', 'time': '0040-04-24 21:00:00'}, {'u': 'trip_0', 'v': 'South Paackland Ferry Terminal', 'time': '0040-04-24 19:00:00'}, {'u': 'trip_0', 'v': 582184557, 'time': '0040-04-24 16:17:00'}, {'u': 'trip_0', 'v': 581853838, 'time': '0040-04-24 16:17:00'}, {'u': 'trip_1', 'v': 'Seal'}, {'u': 'trip_1', 'v': 37830690, 'time': '2040-06-16 06:29:00'}, {'u': 'trip_1', 'v': 37826661, 'time': '2040-06-16 10:38:00'}, {'u': 'trip_1', 'v': 37892464, 'time': '2040-06-16 12:36:00'}, {'u': 'trip_2', 'v': 'Ed Helpsford'}, {'u': 'trip_2', 'v': 950773347, 'time': '2040-06-28 08:23:00'}, {'u': 'trip_3', 'v': 'Teddy Goldstein'}, {'u': 'trip_3', 'v': 'Suna Spit', 'time': '2040-06-06 07:59:00'}, {'u': 'trip_3', 'v': 36988183, 'time': '2040-06-06 07:45:00'}, {'u': 'trip_3', 'v': 35868389, 'time': '2040-06-06 07:28:00'}, {'u': 'trip_3', 'v': 2519420987, 'time': '2040-06-06 07:59:00'}, {'u': 'trip_4', 'v': 'Seal'}, {'u': 'trip_4', 'v'

In [18]:
print(trout_role_set)

[{'u': 'trip_146', 'v': 'Simone Kat'}, {'u': 'trip_146', 'v': 36988183, 'time': '0040-04-06 07:36:00'}, {'u': 'trip_146', 'v': 'Suna Spit', 'time': '0040-04-06 07:36:00'}, {'u': 'trip_146', 'v': 9196630064, 'time': '0040-04-06 06:54:00'}, {'u': 'trip_146', 'v': 30179401, 'time': '0040-04-06 06:54:00'}, {'u': 'trip_146', 'v': 4978801697, 'time': '0040-04-06 06:54:00'}, {'u': 'trip_188', 'v': 'Carol Limpet'}, {'u': 'trip_188', 'v': 'Haacklee Ferry Terminal', 'time': '0040-06-20 21:00:00'}, {'u': 'trip_188', 'v': 'South Paackland Ferry Terminal', 'time': '0040-06-20 19:00:00'}, {'u': 'trip_188', 'v': 582184557, 'time': '0040-06-20 14:13:00'}, {'u': 'trip_188', 'v': 581853838, 'time': '0040-06-20 14:13:00'}, {'u': 'trip_188', 'v': 581717431, 'time': '0040-06-20 14:25:00'}, {'u': 'trip_188', 'v': 582261047, 'time': '0040-06-20 14:38:00'}, {'u': 'trip_188', 'v': 48511194, 'time': '0040-06-20 21:00:00'}, {'u': 'trip_33', 'v': 'Tante Titan'}, {'u': 'trip_33', 'v': 'South Paackland Ferry Termin

## INFERIRE I TIPI DI NODI SENZA TYPE

In [19]:
def untyped_node_edges(G, name):
    """Per ogni nodo senza campo 'type', elenca gli archi in entrata e in uscita
    (con il relativo 'role') per poterne inferire il tipo dalla struttura del grafo."""
    untyped_ids = [n for n, d in G.nodes(data=True) if "type" not in d]
    rows = []
    for uid in untyped_ids:
        incoming = [(u, d.get("role")) for u, v, d in G.in_edges(uid, data=True)]
        outgoing = [(v, d.get("role")) for u, v, d in G.out_edges(uid, data=True)]
        rows.append({
            "dataset": name, "node_id": uid,
            "in_roles": sorted({r for _, r in incoming}),
            "out_roles": sorted({r for _, r in outgoing}),
    
        })
    return pd.DataFrame(rows)

untyped_report = pd.concat(
    [untyped_node_edges(G, name) for name, G in graphs.items()],
    ignore_index=True
)

untyped_report

,dataset,node_id,in_roles,out_roles
0,FILAH,35889363,"[refers_to, travel]",[]
1,FILAH,3753651731,"[refers_to, travel]",[]
2,FILAH,491334376,"[refers_to, travel]",[]
3,FILAH,30192735,"[refers_to, travel]",[]
4,FILAH,35922181,"[refers_to, travel]",[]
5,FILAH,35872967,"[refers_to, travel]",[]
6,FILAH,48468290,"[refers_to, travel]",[]
7,FILAH,9196630064,"[refers_to, travel]",[]
8,FILAH,48475647,"[refers_to, travel]",[]
9,FILAH,581717431,"[refers_to, travel]",[]


## CORREGGERE I NODI SENZA TYPE

In [20]:
manual_type_overrides = {
    "Bay Harvest Corporation": "entity.organization",
    "Harbor Odyssey Tours": "entity.organization",
    "Sean": "entity.person",
}

def infer_and_fix_types(G, name, manual_overrides=manual_type_overrides):
  
    fixed_rows = []
    unresolved = []

    for node_id, data in G.nodes(data=True):
        if "type" in data:
            continue  

        if node_id in manual_overrides:
            inferred_type = manual_overrides[node_id]
        else:
            in_roles = {d.get("role") for _, _, d in G.in_edges(node_id, data=True)}
            if in_roles & {"travel", "refers_to"}:
                inferred_type = "place"
            elif "about" in in_roles:
                inferred_type = "plan"
            else:
                inferred_type = None

        if inferred_type is None:
            unresolved.append(node_id)
            continue

        G.nodes[node_id]["type"] = inferred_type
        G.nodes[node_id]["type_inferred"] = True
        fixed_rows.append({"dataset": name, "node_id": node_id, "type_assegnato": inferred_type})

    if unresolved:
        print(f"[{name}] ATTENZIONE: nodi ancora senza type, da rivedere manualmente: {unresolved}")

    return pd.DataFrame(fixed_rows)

fix_report = pd.concat(
    [infer_and_fix_types(G, name) for name, G in graphs.items()],
    ignore_index=True
)

fix_report

,dataset,node_id,type_assegnato
0,FILAH,35889363,place
1,FILAH,3753651731,place
2,FILAH,491334376,place
3,FILAH,30192735,place
4,FILAH,35922181,place
5,FILAH,35872967,place
6,FILAH,48468290,place
7,FILAH,9196630064,place
8,FILAH,48475647,place
9,FILAH,581717431,place


## MISSING VALUES PT 2

In [21]:

NODE_SCHEMA = {
    "meeting": ["date"],
    "entity.person": ["name", "role"],
    "entity.organization": [],
    "topic": ["short_topic", "long_topic"],
    "discussion": ["short_title", "long_title"],
    "plan": ["short_title", "long_title", "plan_type"],
    "place": ["lat", "lon", "zone"], #zone_detail spessp None
    "trip": ["date", "start", "end"],
}

EDGE_SCHEMA = {
    ("about", "plan"): ["status"],
    ("participant", "entity.person"): ["sentiment", "reason", "industry"],
    ("participant", "entity.organization"): ["sentiment", "reason", "industry"],
    (None, "place"): ["time"],
   
}

# --- Missing sui nodi ---
node_rows = []
for name, G in graphs.items():
    for nid, d in G.nodes(data=True):
        t = d.get("type", "NO_TYPE")
        for field in NODE_SCHEMA.get(t, []):
            if d.get(field) is None:
                node_rows.append({"dataset": name, "tipo": f"node:{t}",
                                   "id": nid, "campo_mancante": field})

missing_nodes = pd.DataFrame(node_rows)

# --- Missing sugli archi ---
edge_rows = []
for name, G in graphs.items():
    for u, v, d in G.edges(data=True):
        role = d.get("role")
        target_type = G.nodes.get(v, {}).get("type", "NO_TYPE")
        for field in EDGE_SCHEMA.get((role, target_type), []):
            if d.get(field) is None:
                edge_rows.append({"dataset": name, "arco": f"{role}:*->{target_type}",
                                   "source": u, "target": v, "campo_mancante": field})

missing_edges = pd.DataFrame(edge_rows)

#stampa
for name in graphs:
    print(f"\n{'='*30} {name} {'='*30}")

    sub_nodes = missing_nodes[missing_nodes.dataset == name]
    print(f"\n-- NODI con campi mancanti ({len(sub_nodes)}) --")
    if sub_nodes.empty:
        print("  nessuno")
    else:
        print(sub_nodes[["tipo", "id", "campo_mancante"]].to_string(index=False))

    sub_edges = missing_edges[missing_edges.dataset == name]
    print(f"\n-- ARCHI con campi mancanti ({len(sub_edges)}) --")
    if sub_edges.empty:
        print("  nessuno")
    else:
        riepilogo = sub_edges.groupby(["arco", "campo_mancante"]).size()
        print(riepilogo.to_string())


============================== FILAH ==============================

-- NODI con campi mancanti (0) --
  nessuno

-- ARCHI con campi mancanti (27) --
arco                                campo_mancante
participant:*->entity.organization  industry          6
                                    reason            6
                                    sentiment         6
participant:*->entity.person        industry          3
                                    reason            3
                                    sentiment         3

============================== TROUT ==============================

-- NODI con campi mancanti (0) --
  nessuno

-- ARCHI con campi mancanti (36) --
arco                                campo_mancante
participant:*->entity.organization  industry          6
                                    reason            6
                                    sentiment         6
participant:*->entity.person        industry          6
                                    

In [22]:
def missing_participants_report(graphs):
    """Stampa gli archi 'participant' (verso persone e organizzazioni)
    che hanno sentiment/reason/industry mancanti, dataset per dataset."""
    rows = []
    for name, G in graphs.items():
        for u, v, d in G.edges(data=True):
            if d.get("role") != "participant":
                continue
            target_type = G.nodes.get(v, {}).get("type", "NO_TYPE")
            if target_type not in ("entity.person", "entity.organization"):
                continue
            campi_mancanti = [f for f in ("sentiment", "reason", "industry") if d.get(f) is None]
            if campi_mancanti:
                rows.append({
                    "dataset": name,
                    "arco": f"participant:*->{target_type}",
                    "discussion_o_plan": u,
                    "target": v,
                    "campi_mancanti": ", ".join(campi_mancanti),
                })

    report = pd.DataFrame(rows)

    for name in graphs:
        sub = report[report.dataset == name]
        print(f"\n{'='*25} {name}: {len(sub)} archi participant incompleti {'='*25}")
        if sub.empty:
            print("  nessuno")
        else:
            for arco_type in sub["arco"].unique():
                sub_type = sub[sub.arco == arco_type]
                print(f"\n  -- {arco_type} ({len(sub_type)}) --")
                print(sub_type[["discussion_o_plan", "target", "campi_mancanti"]]
                      .to_string(index=False))

    return report


missing_participants = missing_participants_report(graphs)


========================= FILAH: 9 archi participant incompleti =========================

  -- participant:*->entity.organization (6) --
                                      discussion_o_plan                  target              campi_mancanti
      deep_fishing_dock_Meeting_2_Importance_Discussion  High Seas Fishing Inc. sentiment, reason, industry
      deep_fishing_dock_Meeting_2_Importance_Discussion Tours Central Ticketing sentiment, reason, industry
deep_fishing_dock_Meeting_3_Maintenance_Plan_Discussion     Industrial Shipping sentiment, reason, industry
           deep_fishing_dock_Meeting_3_Maintenance_Plan     Industrial Shipping sentiment, reason, industry
 deep_fishing_dock_Meeting_4_Findings_Report_Discussion     Industrial Shipping sentiment, reason, industry
            deep_fishing_dock_Meeting_4_Findings_Report     Industrial Shipping sentiment, reason, industry

  -- participant:*->entity.person (3) --
                                                  discussion_o_

### CORREZIONI

In [23]:
def apply_corrections(G):

    # 1) Sean: nome = id (sicuro), ruolo lasciato mancante ma segnalato
    if "Sean" in G.nodes:
        G.nodes["Sean"]["name"] = "Sean"
        G.nodes["Sean"]["role"] = "Uncertain"

    # 2) place 10803677425: recuperato da road_map.json (non inventato)
    if 10803677425 in G.nodes:
        G.nodes[10803677425].update({
            "name": "Harbor Odyssey Tours",
            "lat": 39.09366930039728,   
            "lon": -165.9563594743849,  
            "zone": "tourism",
    
        })

    # 3) plan senza titolo: solo un titolo cosmetico derivato dall'id, marcato come tale
    for nid in ["name_harbor_area_Meeting_11_Harbor_Odyssey_Tours", "concert_Travel_Harborfront_Market"]:
        if nid in G.nodes:
            G.nodes[nid]["short_title"] = nid.replace("_", " ")
            G.nodes[nid]["title_is_derived"] = True  # NON e' il titolo originale

    return G

for G in graphs.values():
    apply_corrections(G)

## I TRE GRAFI SONO VERAMENTE SOTTOINSIEMI?

In [24]:
journalist_nodes = set()
for node, attr in graphs["journalist"].nodes(data=True):
        journalist_nodes.add(node)
        #print(node)
print(journalist_nodes)

{'trip_6', 'trip_21', 'trip_68', 'Industrial Shipping', 'trip_295', 'low_volume_crane_Travel_Harbor_Route_Solutions', 37830690, 'affordable_housing_Meeting_6_Sites_Proposal', 'trip_207', 'expanding_tourist_wharf_Meeting_11_Report_Update_Discussion', 'trip_216', 12698974244, 'deep_fishing_dock', 'waterfront_market_Meeting_7_Benefits_Feasibility', 'trip_143', 9196630064, 36952113, 'trip_81', 48554034, 'trip_183', 'seafood_festival_Meeting_1_Discussion', 'trip_321', 'trip_33', 2519420987, 'trip_63', 'trip_330', 'trip_76', 48580678, 'heritage_walking_tour_Meeting_7_Discussion', 30179401, 'marine_life_deck_Travel_Tropics_Environmental_Hub_Discussion', 'trip_0', 'trip_297', 'low_volume_crane_Meeting_3_Proposal_Discussion', 'trip_211', 'trip_91', 'new_crane_lomark_Meeting_8_Share_Findings', 'waterfront_market_Meeting_9_Report_Discussion', 48590938, 48513116, 48580702, 'trip_59', 'trip_71', 'trip_223', 37826661, 'trip_82', 'trip_176', 'waterfront_market_Meeting_9_Report', 'waterfront_market_Tr

In [25]:
FILAH_nodes = set()
for node, attr in graphs["FILAH"].nodes(data=True):
        FILAH_nodes.add(node)
        #print(node)
print(FILAH_nodes)

{'Industrial Shipping', 'low_volume_crane_Travel_Harbor_Route_Solutions', 'trip_207', 'affordable_housing_Meeting_6_Sites_Proposal', 37830690, 'deep_fishing_dock', 'trip_143', 9196630064, 'trip_81', 36952113, 'trip_183', 'seafood_festival_Meeting_1_Discussion', 2519420987, 'trip_63', 'trip_76', 48580678, 'heritage_walking_tour_Meeting_7_Discussion', 'marine_life_deck_Travel_Tropics_Environmental_Hub_Discussion', 'trip_0', 'trip_211', 48590938, 48580702, 'trip_59', 'trip_71', 'waterfront_market_Travel_Harbor_Edge_Grill', 'trip_144', 'trip_121', 581861511, 'trip_61', 37810321, 37828755, 'marine_life_deck_Travel_Tropics_Environmental_Hub_Completed_Discussion', 'trip_112', 'trip_251', 2133919910, 'trip_186', 'seafood_festival_Travel_Dock_Roll_Hall_of_Fame_Completed_Discussion', 'trip_274', 'trip_40', 'seafood_festival_Travel_Dock_Roll_Hall_of_Fame_Discussion', 'trip_120', 35872967, 'Daughters of Port Grove', 37849292, 'trip_12', 2519421134, 'name_harbor_area', 35889363, 48511187, 48511190,

In [26]:
TROUT_nodes = set()
for node, attr in graphs["TROUT"].nodes(data=True):
        TROUT_nodes.add(node)
        #print(node)
print(TROUT_nodes)

{'affordable_housing_Meeting_9_Invite_Developers_Discussion', 'Industrial Shipping', 'waterfront_market', 'name_inspection_office', 'trip_177', 'Suna Spit', 'statue_john_smoth_Meeting_8_Proposal_Discussion', 'Meeting_3', 'trip_267', 'low_volume_crane_Travel_Harbor_Route_Solutions', 'concert', 'fish_vacuum_Meeting_1_Introduction', 4978801697, 'affordable_housing_Meeting_6_Sites_Proposal', 'marine_life_deck_Meeting_12_Environmental_Impact_Report_Discussion', 'expanding_tourist_wharf_Meeting_11_Report_Update_Discussion', 'expanding_tourist_wharf_Meeting_10_Report_Update_Discussion', 'deep_fishing_dock', 'waterfront_market_Meeting_7_Benefits_Feasibility', 9196630064, 'affordable_housing_Travel_Tidewater_Flats_Discussion', 'affordable_housing_Meeting_8_Gather_Feedback_Discussion', 'Paackland Container Inc.', 'trip_33', 2519420987, 'heritage_walking_tour_Meeting_7_Discussion', 30179401, 'low_volume_crane_Meeting_3_Proposal_Discussion', 'expanding_tourist_wharf_Meeting_7_Initial_Views_Discuss

In [27]:
print(TROUT_nodes.issubset(journalist_nodes))

True


In [28]:
print(FILAH_nodes.issubset(journalist_nodes))

True


verifica se in filah/trout ci sono tutti i topic

In [29]:
def check_topic_coverage(graphs, super_name="journalist"):
    """Verifica se l'insieme dei topic in FILAH/TROUT copre tutti i topic di journalist,
    o se qualcuno manca."""
    topics_super = {n for n, d in graphs[super_name].nodes(data=True) if d.get("type") == "topic"}

    rows = []
    for name, G in graphs.items():
        if name == super_name:
            continue
        topics_sub = {n for n, d in G.nodes(data=True) if d.get("type") == "topic"}
        mancanti = topics_super - topics_sub
        rows.append({
            "dataset": name,
            "n_topic_totali_journalist": len(topics_super),
            "n_topic_presenti": len(topics_sub),
            "n_topic_mancanti": len(mancanti),
            "topic_mancanti": sorted(mancanti) if mancanti else None,
        })
    return pd.DataFrame(rows)


topic_coverage_check = check_topic_coverage(graphs)
topic_coverage_check

,dataset,n_topic_totali_journalist,n_topic_presenti,n_topic_mancanti,topic_mancanti
0,FILAH,15,14,1,[concert]
1,TROUT,15,14,1,[seafood_festival]


verifica se in filah/trout ci sono tutti i meeting

In [30]:
def check_meeting_coverage(graphs, super_name="journalist"):
    """Verifica se l'insieme dei meeting in FILAH/TROUT copre tutti i meeting di journalist,
    o se qualcuno manca."""
    topics_super = {n for n, d in graphs[super_name].nodes(data=True) if d.get("type") == "meeting"}

    rows = []
    for name, G in graphs.items():
        if name == super_name:
            continue
        topics_sub = {n for n, d in G.nodes(data=True) if d.get("type") == "meeting"}
        mancanti = topics_super - topics_sub
        rows.append({
            "dataset": name,
            "n_topic_totali_journalist": len(topics_super),
            "n_topic_presenti": len(topics_sub),
            "n_topic_mancanti": len(mancanti),
            "topic_mancanti": sorted(mancanti) if mancanti else None,
        })
    return pd.DataFrame(rows)


topic_coverage_check = check_meeting_coverage(graphs)
topic_coverage_check

,dataset,n_topic_totali_journalist,n_topic_presenti,n_topic_mancanti,topic_mancanti
0,FILAH,16,12,4,"[Meeting_13, Meeting_14, Meeting_15, Meeting_16]"
1,TROUT,16,13,3,"[Meeting_13, Meeting_14, Meeting_15]"


il codice seguente verifica le proprietà di superset per i missing values: sono presenti negli archi partecipant. Se un arco è presente sia in filah/trout e journalist, campi manacnti devono essere gli stessi in filah/trout e journalist. in particolare per analisi successive vogliamo che che filah/trout non abbia campi mancanti in piu

In [31]:
def check_missing_consistency_edges(graphs, sub_name, super_name="journalist", role="participant"):
    """
    Per gli archi 'participant' presenti sia in sub_name (FILAH/TROUT) sia in
    super_name (journalist), verifica che i campi mancanti (sentiment/reason/industry)
    siano ESATTAMENTE gli stessi nei due dataset. Gli archi assenti in sub_name
    vengono ignorati (non sono una violazione, semplicemente non ci sono).
    """
    def missing_fields_map(G):
        m = {}
        for u, v, d in G.edges(data=True):
            if d.get("role") != role:
                continue
            campi_mancanti = frozenset(
                f for f in ("sentiment", "reason", "industry") if d.get(f) is None
            )
            m[(u, v)] = campi_mancanti
        return m

    sub_map = missing_fields_map(graphs[sub_name])
    super_map = missing_fields_map(graphs[super_name])

    rows = []
    for (u, v), campi_super in super_map.items():
        if (u, v) not in sub_map:
            continue  # l'arco non esiste in FILAH/TROUT: non è confrontabile, non è un errore
        campi_sub = sub_map[(u, v)]
        if campi_sub != campi_super:
            rows.append({
                "dataset": sub_name,
                "discussion_o_plan": u,
                "target": v,
                "campi_mancanti_journalist": sorted(campi_super),
                "campi_mancanti_" + sub_name.lower(): sorted(campi_sub),
            })
    return pd.DataFrame(rows)


for sub_name in ["FILAH", "TROUT"]:
    v_edges = check_missing_consistency_edges(graphs, sub_name)
    print(f"{sub_name}: {len(v_edges)} archi 'participant' con missing diversi da journalist "
          f"(su archi effettivamente presenti in entrambi)")
    if not v_edges.empty:
        print(v_edges.to_string(index=False))

FILAH: 0 archi 'participant' con missing diversi da journalist (su archi effettivamente presenti in entrambi)
TROUT: 0 archi 'participant' con missing diversi da journalist (su archi effettivamente presenti in entrambi)


Per ogni discussion presente nel dataset di parte (FILAH/TROUT), verifica se tutte le sue componenti (plan collegato, topic, place, partecipanti) - così come risultano complete in journalist - sono presenti anche nel sotto-dataset, oppure se ne manca qualche pezzo.

In [32]:
def initiative_completeness_check(G_sub, G_journ, name):
    """Per ogni discussion presente nel dataset di parte, verifica se tutte le sue
    componenti sono presenti anche li'. I partecipanti 'attesi' sono presi
    dall'UNIONE discussion + plan collegato (in journalist) - non solo dalla discussion,
    altrimenti si perdono i partecipanti registrati solo sul plan."""

    rows = []
    discussions_sub = {n for n, d in G_sub.nodes(data=True) if d.get("type") == "discussion"}

    for disc in discussions_sub:
        plan_id = topic_id = place_id = None
        for u, v, d in G_journ.out_edges(disc, data=True):
            if d.get("role") == "about" and G_journ.nodes.get(v, {}).get("type") == "plan":
                plan_id = v
            if d.get("role") == "about" and G_journ.nodes.get(v, {}).get("type") == "topic":
                topic_id = v
            if d.get("role") == "refers_to" and G_journ.nodes.get(v, {}).get("type") == "place":
                place_id = v

        # partecipanti attesi = unione discussion + plan, presi da journalist
        participants_journ = {v for u, v, d in G_journ.out_edges(disc, data=True)
                               if d.get("role") == "participant"}
        if plan_id and plan_id in G_journ.nodes:
            participants_journ |= {v for u, v, d in G_journ.out_edges(plan_id, data=True)
                                    if d.get("role") == "participant"}

        plan_presente = (plan_id in G_sub.nodes) if plan_id else None
        topic_presente = (topic_id in G_sub.nodes) if topic_id else None
        place_presente = (place_id in G_sub.nodes) if place_id else None

        # partecipanti presenti = unione discussion + plan, nel sub-dataset (solo se il plan c'e')
        participants_sub = {v for u, v, d in G_sub.out_edges(disc, data=True)
                             if d.get("role") == "participant"}
        if plan_id and plan_id in G_sub.nodes:
            participants_sub |= {v for u, v, d in G_sub.out_edges(plan_id, data=True)
                                  if d.get("role") == "participant"}

        partecipanti_mancanti = participants_journ - participants_sub
        completa = (
            (plan_presente is not False) and (topic_presente is not False) and
            (place_presente is not False) and (len(partecipanti_mancanti) == 0)
        )

        rows.append({
            "dataset": name, "discussion": disc, "plan_atteso": plan_id, "topic_presente": topic_presente,
            "plan_presente": plan_presente, "n_partecipanti_attesi": len(participants_journ),
            "partecipanti_mancanti": sorted(partecipanti_mancanti) if partecipanti_mancanti else None,
            "discussione_completa": completa,
        })
    return pd.DataFrame(rows)

G_journ = graphs["journalist"]
completeness = pd.concat(
    [initiative_completeness_check(graphs[name], G_journ, name) for name in ["FILAH", "TROUT"]],
    ignore_index=True
)
print(completeness.groupby("dataset")["discussione_completa"].value_counts())
completeness[~completeness["discussione_completa"]]

dataset  discussione_completa
FILAH    True                    45
         False                   13
TROUT    True                    32
         False                    7
Name: count, dtype: int64


,dataset,discussion,plan_atteso,topic_presente,plan_presente,n_partecipanti_attesi,partecipanti_mancanti,discussione_completa
9,FILAH,statue_john_smoth_Meeting_8_Proposal_Discussion,statue_john_smoth_Meeting_8_Proposal,True,True,2,[Tante Titan],False
12,FILAH,affordable_housing_Meeting_7_Debate_Discussion,affordable_housing_Meeting_7_Debate,True,True,2,[Ed Helpsford],False
16,FILAH,name_inspection_office_Meeting_8_Proposal_Disc...,name_inspection_office_Meeting_8_Proposal,True,False,2,[Tante Titan],False
17,FILAH,waterfront_market_Meeting_7_Discussion,waterfront_market_Meeting_7_Benefits_Feasibility,True,False,4,"[Ed Helpsford, Tante Titan]",False
19,FILAH,renaming_park_himark_Meeting_4_Name_Ideas_Disc...,renaming_park_himark_Meeting_4_Name_Ideas,True,True,2,[Tante Titan],False
22,FILAH,affordable_housing_Meeting_8_Gather_Feedback_D...,affordable_housing_Meeting_8_Feedback,True,True,2,[Ed Helpsford],False
23,FILAH,seafood_festival_Meeting_1_Discussion,seafood_festival_Meeting_1_Feasibility,True,True,3,[Tante Titan],False
24,FILAH,deep_fishing_dock_Meeting_4_Findings_Report_Di...,deep_fishing_dock_Meeting_4_Findings_Report,True,True,2,[Teddy Goldstein],False
28,FILAH,name_inspection_office_Travel_Lomark_Civic_Pla...,name_inspection_office_Travel_Lomark_Civic_Plaza,True,True,2,[Tante Titan],False
30,FILAH,expanding_tourist_wharf_Meeting_9_Start_Report...,expanding_tourist_wharf_Meeting_9_Report,True,True,2,[Teddy Goldstein],False


I casi gravi sono quelli in cui manca il plan. Comunque se si considerasse il plan come 'sovrastruttura' in cui incorporare discussion--->cioè Initiative, basterebbe mettere che non è presente in filah/trout con missing_plan_filah e missing_plan_trout. Poi il topic è sempre presente quindi non crea problemi, per le persone basta aggiungere due attributi missing_person_filah e missing_person_trout

Per ogni meeting assente da FILAH/TROUT, sono assenti anche TUTTE le discussion/plan che in journalist sono collegate a quel meeting

In [33]:
def check_missing_meeting_implies_missing_initiatives(graphs, super_name="journalist"):
    """Per ogni meeting assente da FILAH/TROUT, verifica se sono assenti anche TUTTE
    le discussion/plan che in journalist sono collegate a quel meeting (arco part_of)."""
    G_journ = graphs[super_name]

    meeting_initiatives = defaultdict(set)
    for u, v, d in G_journ.edges(data=True):
        if d.get("role") == "part_of":
            meeting_initiatives[u].add(v)

    meetings_super = {n for n, d in G_journ.nodes(data=True) if d.get("type") == "meeting"}

    rows = []
    for name, G_sub in graphs.items():
        if name == super_name:
            continue
        meetings_sub = {n for n, d in G_sub.nodes(data=True) if d.get("type") == "meeting"}
        meeting_mancanti = meetings_super - meetings_sub

        for meeting_id in meeting_mancanti:
            iniziative_attese = meeting_initiatives.get(meeting_id, set())
            iniziative_presenti = {i for i in iniziative_attese if i in G_sub.nodes}
            rows.append({
                "dataset": name, "meeting_mancante": meeting_id,
                "n_iniziative_attese": len(iniziative_attese),
                "n_iniziative_presenti_comunque": len(iniziative_presenti),
                "iniziative_presenti_comunque": sorted(iniziative_presenti) if iniziative_presenti else None,
                "ipotesi_confermata": len(iniziative_presenti) == 0,
            })
    return pd.DataFrame(rows)


check_meetings = check_missing_meeting_implies_missing_initiatives(graphs)
check_meetings

,dataset,meeting_mancante,n_iniziative_attese,n_iniziative_presenti_comunque,iniziative_presenti_comunque,ipotesi_confermata
0,FILAH,Meeting_13,0,0,None,True
1,FILAH,Meeting_15,0,0,None,True
2,FILAH,Meeting_14,4,0,None,True
3,FILAH,Meeting_16,7,0,None,True
4,TROUT,Meeting_13,0,0,None,True
5,TROUT,Meeting_15,0,0,None,True
6,TROUT,Meeting_14,4,0,None,True


In [34]:
def check_missing_topic_implies_missing_initiatives(graphs, super_name="journalist"):
    """Per ogni topic assente da FILAH/TROUT, verifica se sono assenti anche TUTTE
    le discussion/plan che in journalist sono collegate a quel topic."""
    G_journ = graphs[super_name]

    # topic -> insieme di discussion/plan collegate (in journalist)
    topic_initiatives = defaultdict(set)
    for u, v, d in G_journ.edges(data=True):
        if d.get("role") in ("about", "plan") and G_journ.nodes.get(v, {}).get("type") == "topic":
            topic_initiatives[v].add(u)

    topics_super = {n for n, d in G_journ.nodes(data=True) if d.get("type") == "topic"}

    rows = []
    for name, G_sub in graphs.items():
        if name == super_name:
            continue
        topics_sub = {n for n, d in G_sub.nodes(data=True) if d.get("type") == "topic"}
        topic_mancanti = topics_super - topics_sub

        for topic_id in topic_mancanti:
            iniziative_attese = topic_initiatives.get(topic_id, set())
            iniziative_presenti = {i for i in iniziative_attese if i in G_sub.nodes}
            rows.append({
                "dataset": name, "topic_mancante": topic_id,
                "n_iniziative_attese": len(iniziative_attese),
                "n_iniziative_presenti_comunque": len(iniziative_presenti),
                "iniziative_presenti_comunque": sorted(iniziative_presenti) if iniziative_presenti else None,
                "ipotesi_confermata": len(iniziative_presenti) == 0,
            })
    return pd.DataFrame(rows)

#graphs = {name: load_graph(p) for name, p in FILES.items()}
check_result = check_missing_topic_implies_missing_initiatives(graphs)
check_result

,dataset,topic_mancante,n_iniziative_attese,n_iniziative_presenti_comunque,iniziative_presenti_comunque,ipotesi_confermata
0,FILAH,concert,11,0,None,True
1,TROUT,seafood_festival,11,0,None,True


(sopra)topic è sempre true, ma il filah/trout mancano dei topic. se manca il topic, manca tutta la discussione associata


(sotto)Per ogni meeting presente sia nel dataset di parte sia in journalist, confronta
    l'insieme di discussion/plan collegati (arco part_of) nei due dataset.

In [35]:
def check_meeting_initiative_alignment(graphs, super_name="journalist"):
    """Per ogni meeting presente sia nel dataset di parte sia in journalist, confronta
    l'insieme di discussion/plan collegati (arco part_of) nei due dataset."""
    G_journ = graphs[super_name]

    def meeting_initiatives_map(G):
        m = defaultdict(set)
        for u, v, d in G.edges(data=True):
            if d.get("role") == "part_of":
                m[u].add(v)
        return m

    init_journ = meeting_initiatives_map(G_journ)

    rows = []
    for name, G_sub in graphs.items():
        if name == super_name:
            continue
        init_sub = meeting_initiatives_map(G_sub)

        meetings_journ = {n for n, d in G_journ.nodes(data=True) if d.get("type") == "meeting"}
        meetings_sub = {n for n, d in G_sub.nodes(data=True) if d.get("type") == "meeting"}
        meetings_common = meetings_journ & meetings_sub

        for m in sorted(meetings_common):
            attese = init_journ.get(m, set())
            presenti = init_sub.get(m, set())
            mancanti = attese - presenti
            extra = presenti - attese  # non dovrebbe mai capitare (gia' verificato in generale)
            rows.append({
                "dataset": name, "meeting": m,
                "n_iniziative_journalist": len(attese),
                "n_iniziative_presenti": len(presenti),
                "n_iniziative_mancanti": len(mancanti),
                "iniziative_mancanti": sorted(mancanti) if mancanti else None,
                "iniziative_extra_INASPETTATE": sorted(extra) if extra else None,
            })
    return pd.DataFrame(rows)


meeting_alignment = check_meeting_initiative_alignment(graphs)
meeting_alignment

,dataset,meeting,n_iniziative_journalist,n_iniziative_presenti,n_iniziative_mancanti,iniziative_mancanti,iniziative_extra_INASPETTATE
0,FILAH,Meeting_1,4,4,0,None,None
1,FILAH,Meeting_10,11,4,7,[affordable_housing_Meeting_10_Housing_Proposa...,None
2,FILAH,Meeting_11,11,4,7,[expanding_tourist_wharf_Meeting_11_Report_Upd...,None
3,FILAH,Meeting_12,9,3,6,[marine_life_deck_Meeting_12_Environmental_Imp...,None
4,FILAH,Meeting_2,9,9,0,None,None
5,FILAH,Meeting_3,14,8,6,"[fish_vacuum_Meeting_3_Report, fish_vacuum_Mee...",None
6,FILAH,Meeting_4,13,7,6,"[deep_fishing_dock_Meeting_4_Recommendation, d...",None
7,FILAH,Meeting_5,10,8,2,[new_crane_lomark_Meeting_5_Cost_Impact_Report...,None
8,FILAH,Meeting_6,2,2,0,None,None
9,FILAH,Meeting_7,17,10,7,"[concert_Meeting_7_Concert_Issues, concert_Mee...",None


## QUANTI PLACE/PERSON SONO CONNESSI A CIASCUN TRIP?

In [36]:

def trip_place_counts(G, name, type):
    """Per ogni trip, conta quante place/person sono collegate (archi senza role, target di tipo 'place')."""
    trip_places = defaultdict(list)
    for u, v, d in G.edges(data=True):
        if d.get("role") is None and G.nodes.get(u, {}).get("type") == "trip":
            if G.nodes.get(v, {}).get("type") == type:
                trip_places[u].append(v)

    rows = [{"dataset": name, "trip_id": trip_id, "n_place": len(places)}
            for trip_id, places in trip_places.items()]
    return pd.DataFrame(rows)

trip_place_df = pd.concat(
    [trip_place_counts(G, name, "place") for name, G in graphs.items()],
    ignore_index=True
)

trip_person_df = pd.concat(
    [trip_place_counts(G, name, "entity.person") for name, G in graphs.items()],
    ignore_index=True
)

#print(trip_place_df.groupby("dataset")["n_place"].value_counts().sort_index())
#print()
print("massimo numero di place per trip, per dataset:")
print(trip_place_df.groupby("dataset")["n_place"].max())
print()
print("massimo numero di person per trip, per dataset:")
print(trip_person_df.groupby("dataset")["n_place"].max())


massimo numero di place per trip, per dataset:
dataset
FILAH          2
TROUT          8
journalist    12
Name: n_place, dtype: int64

massimo numero di person per trip, per dataset:
dataset
FILAH         1
TROUT         1
journalist    1
Name: n_place, dtype: int64


## QUANTI TOPIC E PLACE PER DISCUSSION/PLAN?

In [37]:

def max_targets_per_source(G, role, target_type):
    """Per ogni nodo sorgente, conta quanti target distinti di un certo tipo raggiunge
    tramite un arco con quel 'role'. Utile per verificare se la relazione e' davvero 1:1."""
    from collections import defaultdict
    targets = defaultdict(set)
    for u, v, d in G.edges(data=True):
        if d.get("role") == role and G.nodes.get(v, {}).get("type") == target_type:
            targets[u].add(v)
    return targets


def check_cardinality(graphs, role, target_type, source_type=None):
    rows = []
    for name, G in graphs.items():
        targets = max_targets_per_source(G, role, target_type)
        for source_id, tset in targets.items():
            if source_type and G.nodes.get(source_id, {}).get("type") != source_type:
                continue
            rows.append({"dataset": name, "source": source_id, "n_target": len(tset),
                         "targets": sorted(tset) if len(tset) > 1 else None})
    return pd.DataFrame(rows)


checks = {
    "discussion -> topic (about)": ("about", "topic", "discussion"),
    "discussion -> place (refers_to)": ("refers_to", "place", "discussion"),
    "plan -> topic (plan)": ("plan", "topic", "plan"),
    "plan -> place (travel)": ("travel", "place", "plan"),
}

for label, (role, target_type, source_type) in checks.items():
    df = check_cardinality(graphs, role, target_type, source_type)
    print(f"--- {label} ---")
    print(df["n_target"].value_counts().sort_index())
    multi = df[df["n_target"] > 1]
    if not multi.empty:
        print("ATTENZIONE, sorgenti con più di un target:")
        print(multi.to_string(index=False))
    print()


--- discussion -> topic (about) ---
n_target
1    198
Name: count, dtype: int64

--- discussion -> place (refers_to) ---
n_target
1    76
Name: count, dtype: int64

--- plan -> topic (plan) ---
n_target
1    146
Name: count, dtype: int64

--- plan -> place (travel) ---
n_target
1    41
Name: count, dtype: int64



In [38]:
def discussion_plan_links(G):
    """Ritorna la lista di archi (discussion_id, plan_id, status) per il grafo G."""
    return [
        (u, v, d.get("status"))
        for u, v, d in G.edges(data=True)
        if d.get("role") == "about" and G.nodes.get(v, {}).get("type") == "plan"
    ]
def single_target(targets_map, node_id):
    """Estrae l'unico target (o None) da max_targets_per_source, per un nodo dato."""
    tset = targets_map.get(node_id, set())
    return next(iter(tset)) if tset else None


def check_topic_consistency(G, name):
    disc_topic_map = max_targets_per_source(G, "about", "topic")
    plan_topic_map = max_targets_per_source(G, "plan", "topic")

    rows = []
    for disc, plan, status in discussion_plan_links(G):
        topic_disc = single_target(disc_topic_map, disc)
        topic_plan = single_target(plan_topic_map, plan)
        rows.append({
            "dataset": name, "discussion": disc, "plan": plan,
            "topic_discussion": topic_disc, "topic_plan": topic_plan,
            # coerente se sono uguali, oppure se uno dei due manca (non e' un conflitto, solo un dato assente)
            "topic_coerente": (topic_disc is None or topic_plan is None or topic_disc == topic_plan),
        })
    return pd.DataFrame(rows)


topic_consistency = pd.concat(
    [check_topic_consistency(G, name) for name, G in graphs.items()],
    ignore_index=True
)

print(topic_consistency.groupby("dataset")["topic_coerente"].value_counts())
topic_consistency[~topic_consistency["topic_coerente"]]  # eventuali veri conflitti (entrambi presenti ma diversi)

dataset     topic_coerente
FILAH       True              53
TROUT       True              36
journalist  True              98
Name: count, dtype: int64


,dataset,discussion,plan,topic_discussion,topic_plan,topic_coerente


## INDUSTRY è SOVRAINSIEME DI TOPIC? 

In [39]:
def topic_of_node(G, node_id):
    """Topic collegato a una discussion (about->topic) o a un plan (plan->topic)."""
    for _, v, d in G.out_edges(node_id, data=True):
        if d.get("role") in ("about", "plan") and G.nodes.get(v, {}).get("type") == "topic":
            return v
    return None


def check_topic_industry_coherence(G):
    """Per ogni topic, raccoglie tutti i set di industry osservati sui suoi partecipanti
    (in tutte le discussion/plan collegate). Distingue industry=None (dato mancante,
    da IGNORARE nel controllo di coerenza) da industry=[]."""
    topic_industries = defaultdict(set)
    for u, v, d in G.edges(data=True):
        if d.get("role") != "participant":
            continue
        topic_id = topic_of_node(G, u)
        raw = d.get("industry")
        if raw is None:
            industry = "MISSING"
        else:
            industry = tuple(sorted(raw)) if raw else "EMPTY_LEGIT"
        topic_industries[topic_id].add(industry)

    rows = []
    for topic_id, observed in topic_industries.items():
        informative = {x for x in observed if x != "MISSING"}
        rows.append({
            "topic_id": topic_id, "industry_osservate": observed,
            "coerente": len(informative) <= 1,
        })
    return pd.DataFrame(rows)


coherence = check_topic_industry_coherence(G_journ)
print(coherence["coerente"].value_counts())
coherence[~coherence["coerente"]]

coerente
True     14
False     1
Name: count, dtype: int64


,topic_id,industry_osservate,coerente
4,fish_vacuum,"{(large vessel,), (small vessel,), MISSING}",False


in quasi tutti i casi industry è sovrainsieme di topic, tranne per fish vacuum che è collegato sia a small vessel che large vessel

In [40]:
def print_participants_for_topic(G, topic_id):
    """Stampa tutti i partecipanti (persona/org, sentiment, reason, industry) di
    tutte le discussion/plan collegate a un dato topic."""
    # trovo tutte le discussion/plan che hanno questo topic (via about o plan)
    initiatives_for_topic = [
        u for u, v, d in G.edges(data=True)
        if d.get("role") in ("about", "plan") and v == topic_id
        and G.nodes.get(u, {}).get("type") in ("discussion", "plan")
    ]

    for source in initiatives_for_topic:
        print(f"--- {source} ({G.nodes[source].get('type')}) ---")
        for _, target, d in G.out_edges(source, data=True):
            if d.get("role") == "participant":
                print(f"  {target:25s} sentiment={d.get('sentiment')}  "
                      f"industry={d.get('industry')}")
                print(f"      reason: {d.get('reason')}")
        print()


print_participants_for_topic(G_journ, "fish_vacuum")

--- fish_vacuum_Meeting_1_Introduction_Discussion (discussion) ---
  Teddy Goldstein           sentiment=0.5  industry=['large vessel']
      reason: Useful for fishing operations but may have environmental concerns.
  High Seas Fishing Inc.    sentiment=1  industry=['small vessel']
      reason: Small vessels are the backbone of the company, and this improves operations.
  Bay Harvest Corporation   sentiment=None  industry=None
      reason: None

--- fish_vacuum_Meeting_1_Introduction (plan) ---
  Seal                      sentiment=0  industry=['large vessel']
      reason: Balanced view on new technology applications.
  High Seas Fishing Inc.    sentiment=1  industry=['small vessel']
      reason: Small vessels are the backbone of the company, and this improves operations.
  Bay Harvest Corporation   sentiment=None  industry=None
      reason: None

--- fish_vacuum_Travel_Bay_Harvest_Corporation_Discussion (discussion) ---
  Simone Kat                sentiment=-0.1  industry=['larg

Leggendo i reason: "Small vessels are the backbone of the company" (High Seas Fishing Inc.) e "Useful for fishing operations" (Teddy Goldstein, large vessel) — non sono due persone in disaccordo sulla stessa etichetta, sono due punti di vista genuinamente diversi sulla stessa tecnologia: un "fish vacuum" può essere rilevante sia per la piccola pesca sia per la grande pesca, e ciascun partecipante commenta dalla prospettiva del proprio settore.

Un topic può essere legato a più industrie contemporaneamente (come fish_vacuum, rilevante per entrambe), e ogni riga participant riporta quale industria è rilevante in quel caso per quel partecipante specifico

## RIDONDANZA MEETING DISCUSSION PLAN

In [41]:
def discussion_plan_links(G):
    """Ritorna la lista di archi (discussion_id, plan_id, status) per il grafo G."""
    return [
        (u, v, d.get("status"))
        for u, v, d in G.edges(data=True)
        if d.get("role") == "about" and G.nodes.get(v, {}).get("type") == "plan"
    ]

In [42]:
def meeting_of_map(G):
    """id (discussion o plan) -> set di meeting collegati tramite arco part_of.
    Utile anche per verificare che non ce ne sia più di uno (dovrebbe essere sempre 1)."""
    m = defaultdict(set)
    for u, v, d in G.edges(data=True):
        if d.get("role") == "part_of":
            m[v].add(u)
    return m

In [43]:
def check_meeting_redundancy_detailed(G, name):
    """Controllo a doppio livello:
    - per COPPIA (discussion, plan): il meeting coincide esattamente?
    - per PLAN aggregato: il meeting del plan è comunque raggiungibile tramite
      ALMENO UNA delle sue discussion (anche se non questa specifica)?
    """
    meeting_of = meeting_of_map(G)
    plan_discs = defaultdict(list)
    for disc, plan, status in discussion_plan_links(G):
        plan_discs[plan].append(disc)

    rows = []
    for disc, plan, status in discussion_plan_links(G):
        plan_meeting = next(iter(meeting_of.get(plan, set())), None)
        disc_meeting = next(iter(meeting_of.get(disc, set())), None)

        # e' recuperabile guardando TUTTE le discussion di questo plan, non solo questa?
        all_disc_meetings = set()
        for d in plan_discs[plan]:
            all_disc_meetings |= meeting_of.get(d, set())

        rows.append({
            "dataset": name, "plan": plan, "discussion": disc, "status": status,
            "meeting_plan": plan_meeting,
            "meeting_QUESTA_discussion": disc_meeting,
            "coppia_coincide": disc_meeting == plan_meeting,
            "recuperabile_da_ALTRA_discussion_dello_stesso_plan": (
                plan_meeting in (all_disc_meetings - {disc_meeting} if disc_meeting == plan_meeting else all_disc_meetings)
            ),
        })
    return pd.DataFrame(rows)


meeting_detail = pd.concat(
    [check_meeting_redundancy_detailed(G, name) for name, G in graphs.items()],
    ignore_index=True
)
print(meeting_detail.groupby("dataset")["coppia_coincide"].value_counts())
meeting_detail[~meeting_detail["coppia_coincide"]]  # tutte le coppie dove NON coincide, plan per plan

dataset     coppia_coincide
FILAH       True               40
            False              13
TROUT       True               31
            False               5
journalist  True               72
            False              26
Name: count, dtype: int64


,dataset,plan,discussion,status,meeting_plan,meeting_QUESTA_discussion,coppia_coincide,recuperabile_da_ALTRA_discussion_dello_stesso_plan
1,FILAH,expanding_tourist_wharf_Travel_Harbor_Route_So...,expanding_tourist_wharf_Travel_Harbor_Route_So...,completed,Meeting_8,Meeting_9,False,True
7,FILAH,deep_fishing_dock_Travel_High_Seas_Fishing_Inc,deep_fishing_dock_Travel_High_Seas_Fishing_Inc...,completed,Meeting_3,Meeting_4,False,True
11,FILAH,new_crane_lomark_Travel_Blue_Wave_Shipping,new_crane_lomark_Travel_Blue_Wave_Shipping_Com...,completed,Meeting_5,Meeting_8,False,True
15,FILAH,fish_vacuum_Travel_Bay_Harvest_Corporation,fish_vacuum_Travel_Bay_Harvest_Corporation_Com...,completed,Meeting_2,Meeting_3,False,True
17,FILAH,low_volume_crane_Travel_Harbor_Route_Solutions,low_volume_crane_Travel_Harbor_Route_Solutions...,completed,Meeting_5,Meeting_8,False,True
21,FILAH,low_volume_crane_Travel_Harbor_Builders,low_volume_crane_Travel_Harbor_Builders_Comple...,completed,Meeting_8,Meeting_9,False,True
22,FILAH,low_volume_crane_Travel_Haacklee_Assembly_Co,low_volume_crane_Travel_Haacklee_Assembly_Co_C...,completed,Meeting_8,Meeting_9,False,True
31,FILAH,renaming_park_himark_Travel_Himark_City_Hall,renaming_park_himark_Travel_Himark_City_Hall_C...,completed,Meeting_5,Meeting_8,False,True
34,FILAH,name_inspection_office_Travel_Lomark_Civic_Plaza,name_inspection_office_Travel_Lomark_Civic_Pla...,completed,Meeting_8,Meeting_9,False,False
38,FILAH,marine_life_deck_Travel_Tropics_Environmental_Hub,marine_life_deck_Travel_Tropics_Environmental_...,completed,Meeting_11,Meeting_12,False,True


## DISCUSSION & PLAN

In [44]:
def discussion_plan_cardinality(G, name):
    """Per ogni discussion, conta a quanti plan e' collegata (arco discussion --about--> plan)."""
    disc_to_plans = defaultdict(list)
    for u, v, d in G.edges(data=True):
        if d.get("role") == "about" and G.nodes.get(v, {}).get("type") == "plan":
            disc_to_plans[u].append(v)
    return pd.DataFrame([
        {"dataset": name, "discussion": disc, "n_plan": len(plans), "plans": plans}
        for disc, plans in disc_to_plans.items()
    ])

cardinality = pd.concat(
    [discussion_plan_cardinality(G, name) for name, G in graphs.items()],
    ignore_index=True
)

print(cardinality.groupby("dataset")["n_plan"].value_counts())
cardinality[cardinality["n_plan"] > 1]  # eventuali discussion collegate a più di un plan

dataset     n_plan
FILAH       1         53
TROUT       1         36
journalist  1         98
Name: count, dtype: int64


,dataset,discussion,n_plan,plans


quindi, ogni discussion ha un solo plan. per cui si potrebbero unire le due cose

verficihiamo intanto se  discussion—->entity.person, discussion —> entity.organization (arco 'partecipant') sono sottoinsiemi di  plan —->entity.organization, plan—>entity.person del plan associato a discussion (cioè verificare corenza nei dati)

In [45]:
def participant_set_by_type(G, entity_type):
    """node_id -> set di (target, sentiment, tuple(industry)) per participant verso un tipo di entita' specifico."""
    p = defaultdict(set)
    for u, v, d in G.edges(data=True):
        if d.get("role") == "participant" and G.nodes.get(v, {}).get("type") == entity_type:
            p[u].add((v, d.get("sentiment"), tuple(d.get("industry") or [])))
    return p

def check_subset_by_type(G, name, entity_type):
    dp_links = discussion_plan_links(G)  # (discussion, plan, status), definita in Sezione 7
    part = participant_set_by_type(G, entity_type)
    rows = []
    for disc, plan, status in dp_links: #per ogni disc, plan uniti da about
        d_set = part.get(disc, set()) #prendi da disc la persona/organizzazione legata a disc tramite partecipant
        p_set = part.get(plan, set()) #prendi da plan la persona/organizzazione legata a plan tramite partecipant
        rows.append({
            "dataset": name, "entity_type": entity_type,
            "discussion": disc, "plan": plan, "status": status,
            "n_discussion": len(d_set), "n_plan": len(p_set),
            "discussion_subset_of_plan": d_set <= p_set, #verifica che le persone/org coinvolte nella discussione siano subset di quele coinvolte nel plan
        })
    return pd.DataFrame(rows)

subset_by_type = pd.concat(
    [check_subset_by_type(G, name, etype)
     for name, G in graphs.items()
     for etype in ["entity.person", "entity.organization"]],
    ignore_index=True
)

print(subset_by_type.groupby(["dataset", "entity_type"])["discussion_subset_of_plan"].value_counts())
subset_by_type[~subset_by_type["discussion_subset_of_plan"]]  # eventuali casi incoerenti, per tipo di entita'

dataset     entity_type          discussion_subset_of_plan
FILAH       entity.organization  True                         53
            entity.person        True                         50
                                 False                         3
TROUT       entity.organization  True                         36
            entity.person        True                         34
                                 False                         2
journalist  entity.organization  True                         97
                                 False                         1
            entity.person        True                         88
                                 False                        10
Name: count, dtype: int64


,dataset,entity_type,discussion,plan,status,n_discussion,n_plan,discussion_subset_of_plan
34,FILAH,entity.person,name_inspection_office_Travel_Lomark_Civic_Pla...,name_inspection_office_Travel_Lomark_Civic_Plaza,completed,1,0,False
40,FILAH,entity.person,seafood_festival_Meeting_1_Discussion,seafood_festival_Meeting_1_Feasibility,completed,2,1,False
42,FILAH,entity.person,seafood_festival_Meeting_2_Feedback_Discussion,seafood_festival_Meeting_2_Feedback,in_progress,1,1,False
114,TROUT,entity.person,affordable_housing_Travel_Waveside_Townhomes_C...,affordable_housing_Travel_Waveside_Townhomes,completed,1,0,False
137,TROUT,entity.person,fish_vacuum_Meeting_1_Introduction_Discussion,fish_vacuum_Meeting_1_Introduction,completed,1,1,False
186,journalist,entity.person,statue_john_smoth_Travel_The_Bait_Stich_Comple...,statue_john_smoth_Travel_The_Bait_Stich,completed,1,1,False
198,journalist,entity.person,fish_vacuum_Meeting_1_Introduction_Discussion,fish_vacuum_Meeting_1_Introduction,completed,1,1,False
214,journalist,entity.person,affordable_housing_Travel_Waveside_Townhomes_C...,affordable_housing_Travel_Waveside_Townhomes,completed,1,1,False
238,journalist,entity.person,name_harbor_area_Meeting_14_Finalize_Name_Disc...,name_harbor_area_Meeting_14_Finalize_Name,completed,1,1,False
240,journalist,entity.person,name_inspection_office_Meeting_8_Proposal_Disc...,name_inspection_office_Meeting_8_Proposal,planned,1,1,False


In [46]:
def inconsistent_person_details(G, name):
    dp_links = discussion_plan_links(G)
    part = participant_set_by_type(G, "entity.person")
    rows = []
    for disc, plan, status in dp_links:
        d_set = part.get(disc, set())
        p_set = part.get(plan, set())
        if d_set <= p_set:
            continue  # coerente, salta

        persone_discussion = {t for (t, s, ind) in d_set} # d_set = (v, sentiment, industry)
        persone_plan = {t for (t, s, ind) in p_set}
        sentiment_discussion = {s for (t, s, ind) in d_set}
        sentiment_plan = {s for (t, s, ind) in p_set}
        industry_discussion = {ind for (t, s, ind) in d_set}
        industry_plan = {ind for (t, s, ind) in p_set}

        rows.append({
            "dataset": name, "discussion": disc, "plan": plan, "status": status,
            "persone_in_discussion": persone_discussion,
            "persone_in_plan": persone_plan,
            "solo_in_discussion": sorted(set(persone_discussion) - set(persone_plan)),
            "sentiment_discussion": sentiment_discussion,# sorted(d_set),
            "sentiment_plan": sentiment_plan,
            "industry_discussion": industry_discussion,
            "industry_plan": industry_plan
        })
    return pd.DataFrame(rows)

inconsistent_persons = pd.concat(
    [inconsistent_person_details(G, name) for name, G in graphs.items()],
    ignore_index=True
)

inconsistent_persons

,dataset,discussion,plan,status,persone_in_discussion,persone_in_plan,solo_in_discussion,sentiment_discussion,sentiment_plan,industry_discussion,industry_plan
0,FILAH,name_inspection_office_Travel_Lomark_Civic_Pla...,name_inspection_office_Travel_Lomark_Civic_Plaza,completed,{Simone Kat},{},[Simone Kat],{0},{},{()},{}
1,FILAH,seafood_festival_Meeting_1_Discussion,seafood_festival_Meeting_1_Feasibility,completed,"{Simone Kat, Carol Limpet}",{Simone Kat},[Carol Limpet],{0.75},{0.75},"{(tourism, small vessel)}","{(tourism, small vessel)}"
2,FILAH,seafood_festival_Meeting_2_Feedback_Discussion,seafood_festival_Meeting_2_Feedback,in_progress,{Carol Limpet},{Simone Kat},[Carol Limpet],{0.75},{0.75},"{(tourism, small vessel)}","{(tourism, small vessel)}"
3,TROUT,affordable_housing_Travel_Waveside_Townhomes_C...,affordable_housing_Travel_Waveside_Townhomes,completed,{Ed Helpsford},{},[Ed Helpsford],{1},{},"{(large vessel, small vessel)}",{}
4,TROUT,fish_vacuum_Meeting_1_Introduction_Discussion,fish_vacuum_Meeting_1_Introduction,completed,{Teddy Goldstein},{Seal},[Teddy Goldstein],{0.5},{0},"{(large vessel,)}","{(large vessel,)}"
5,journalist,statue_john_smoth_Travel_The_Bait_Stich_Comple...,statue_john_smoth_Travel_The_Bait_Stich,completed,{Tante Titan},{Seal},[Tante Titan],{1},{0.2},{()},{()}
6,journalist,fish_vacuum_Meeting_1_Introduction_Discussion,fish_vacuum_Meeting_1_Introduction,completed,{Teddy Goldstein},{Seal},[Teddy Goldstein],{0.5},{0},"{(large vessel,)}","{(large vessel,)}"
7,journalist,affordable_housing_Travel_Waveside_Townhomes_C...,affordable_housing_Travel_Waveside_Townhomes,completed,{Ed Helpsford},{Simone Kat},[Ed Helpsford],{1},{-1},"{(large vessel, small vessel)}","{(large vessel, small vessel)}"
8,journalist,name_harbor_area_Meeting_14_Finalize_Name_Disc...,name_harbor_area_Meeting_14_Finalize_Name,completed,{Sean},{Tante Titan},[Sean],{None},{1},{()},{()}
9,journalist,name_inspection_office_Meeting_8_Proposal_Disc...,name_inspection_office_Meeting_8_Proposal,planned,{Simone Kat},{Tante Titan},[Simone Kat],{0},{1},{()},{()}


In [47]:
def single_relation(G, role, target_type):
    """node_id -> target_id, per relazioni dove ogni nodo punta al massimo a UN target
    di quel tipo (es. discussion->topic, plan->place)."""
    r = {}
    for u, v, d in G.edges(data=True):
        if d.get("role") == role and G.nodes.get(v, {}).get("type") == target_type:
            r[u] = v
    return r


def check_topic_place_consistency(G, name):
    dp_links = discussion_plan_links(G)

    disc_topic = single_relation(G, "about", "topic")
    plan_topic = single_relation(G, "plan", "topic")
    disc_place = single_relation(G, "refers_to", "place")
    plan_place = single_relation(G, "travel", "place")

    rows = []
    for disc, plan, status in dp_links:
        dt, pt = disc_topic.get(disc), plan_topic.get(plan)
        dpl, ppl = disc_place.get(disc), plan_place.get(plan)
        rows.append({
            "dataset": name, "discussion": disc, "plan": plan,
            "topic_discussion": dt, "topic_plan": pt,
            # coerente se uguali, oppure se uno dei due manca (non confrontabile, non e' un conflitto)
            "topic_coerente": (dt is None or pt is None or dt == pt),
            "place_discussion": dpl, "place_plan": ppl,
            "place_coerente": (dpl is None or ppl is None or dpl == ppl),
        })
    return pd.DataFrame(rows)


topic_place_checks = pd.concat(
    [check_topic_place_consistency(G, name) for name, G in graphs.items()],
    ignore_index=True
)

print(topic_place_checks.groupby("dataset")["topic_coerente"].value_counts())
print()
print(topic_place_checks.groupby("dataset")["place_coerente"].value_counts())
print()
# eventuali conflitti reali (entrambi presenti ma diversi)
topic_place_checks[~topic_place_checks.topic_coerente | ~topic_place_checks.place_coerente]

dataset     topic_coerente
FILAH       True              53
TROUT       True              36
journalist  True              98
Name: count, dtype: int64

dataset     place_coerente
FILAH       True              53
TROUT       True              36
journalist  True              97
            False              1
Name: count, dtype: int64



,dataset,discussion,plan,topic_discussion,topic_plan,topic_coerente,place_discussion,place_plan,place_coerente
178,journalist,waterfront_market_Travel_Harborfront_Market_Co...,waterfront_market_Travel_Harborfront_Market,waterfront_market,waterfront_market,True,37831107.0,35913119.0,False


attenzione questo potrebbe essere un errore, perché il plan fa riferimento a un mercato sul lungomare (Harborfront Market) mentre la discussione riguarda una scuola elementare (Paakland Elementary)

## SENTIMENT

Sentiment: discussion vs plan, per le persone/org in comune

quando la stessa persona compare sia nella discussion sia nel plan, il sentiment è sempre identico

In [48]:
def participant_sentiment_map(G, node_id):
    """id -> sentiment, per gli archi participant uscenti da un nodo (discussion o plan)."""
    return {v: d.get("sentiment") for _, v, d in G.out_edges(node_id, data=True)
            if d.get("role") == "participant"}


def sentiment_match_discussion_plan(G, name):
    """Per ogni coppia (discussion, plan) collegata, confronta il sentiment SOLO per le
    persone/organizzazioni presenti in ENTRAMBI (non l'intero insieme, solo l'overlap)."""
    rows = []
    for disc, plan, status in discussion_plan_links(G):
        d_sent = participant_sentiment_map(G, disc)
        p_sent = participant_sentiment_map(G, plan)
        for entity in set(d_sent) & set(p_sent):
            rows.append({
                "dataset": name, "discussion": disc, "plan": plan, "entity": entity,
                "sentiment_discussion": d_sent[entity], "sentiment_plan": p_sent[entity],
                "match": d_sent[entity] == p_sent[entity],
            })
    return pd.DataFrame(rows)


sentiment_match = pd.concat(
    [sentiment_match_discussion_plan(G, name) for name, G in graphs.items()],
    ignore_index=True
)
print(sentiment_match.groupby("dataset")["match"].value_counts())
sentiment_match[~sentiment_match["match"]]  # eventuali mismatch reali

dataset     match
FILAH       True      57
TROUT       True      43
journalist  True     105
Name: count, dtype: int64


,dataset,discussion,plan,entity,sentiment_discussion,sentiment_plan,match


Sentiment coerente tra checkpoint diversi dello stesso plan. Nessuno "cambia idea" tra un checkpoint e l'altro.

In [49]:
def sentiment_consistency_across_checkpoints(G, name):
    """Per i plan con piu' di una discussion collegata (checkpoint temporali), confronta
    il sentiment della stessa persona/org TRA le diverse discussion (non col plan)."""
    plan_discs = defaultdict(list)
    for disc, plan, status in discussion_plan_links(G):
        plan_discs[plan].append(disc)

    rows = []
    for plan, discs in plan_discs.items():
        if len(discs) < 2:
            continue
        sent_maps = {disc: participant_sentiment_map(G, disc) for disc in discs}
        entities = set().union(*sent_maps.values())
        for entity in entities:
            appearances = {disc: m[entity] for disc, m in sent_maps.items() if entity in m}
            if len(appearances) < 2:
                continue  # compare in un solo checkpoint, niente da confrontare
            rows.append({
                "dataset": name, "plan": plan, "entity": entity,
                "n_checkpoint": len(appearances),
                "sentiment_per_checkpoint": appearances,
                "consistente": len(set(appearances.values())) == 1,
            })
    return pd.DataFrame(rows)


checkpoint_consistency = pd.concat(
    [sentiment_consistency_across_checkpoints(G, name) for name, G in graphs.items()],
    ignore_index=True
)
print(checkpoint_consistency.groupby("dataset")["consistente"].value_counts())
checkpoint_consistency[~checkpoint_consistency["consistente"]]  # eventuali persone che cambiano idea tra un checkpoint e l'altro

dataset     consistente
FILAH       True           12
TROUT       True            3
journalist  True           18
Name: count, dtype: int64


,dataset,plan,entity,n_checkpoint,sentiment_per_checkpoint,consistente


Il sentiment è davvero una funzione stabile di (persona, topic)!

In [50]:
def initiative_of(G, node):
    """Dato un nodo discussion o plan, restituisce l'id del plan associato.
    Discussion e plan che si riferiscono allo stesso evento condividono la
    stessa 'iniziativa': la discussion punta al plan tramite un arco 'about'."""
    node_type = G.nodes[node].get("type")
    if node_type == "plan":
        return node
    if node_type == "discussion":
        for _, target, data in G.out_edges(node, data=True):
            if data.get("role") == "about":
                return target
    return None

rows = []
for u, v, d in G_journ.edges(data=True):
    if d.get("role") != "participant":
        continue
    initiative_id = initiative_of(G_journ, u)
    topic_id = topic_of_node(G_journ, initiative_id) or topic_of_node(G_journ, u)
    raw_industry = d.get("industry")
    rows.append({
        "entity_id": v, "topic_id": topic_id, "initiative_id": initiative_id,
        "sentiment": d.get("sentiment"),
        "reason": d.get("reason"),
        "industry": tuple(sorted(raw_industry)) if raw_industry is not None else None,
    })

df = pd.DataFrame(rows)
# deduplico entro la stessa iniziativa (discussion+plan sono lo stesso evento, gia' verificato)
df = df.drop_duplicates(subset=["entity_id", "topic_id", "initiative_id", "sentiment", "reason", "industry"])

counts = df.groupby(["entity_id", "topic_id"])["initiative_id"].nunique()
multi_pairs = counts[counts > 1].index
print("coppie (entita, topic) con piu' di un'iniziativa distinta:", len(multi_pairs))

check_rows = []
for entity_id, topic_id in multi_pairs:
    sub = df[(df.entity_id == entity_id) & (df.topic_id == topic_id)]
    sentiments = sorted(sub["sentiment"].dropna().unique())
    reasons = sorted(sub["reason"].dropna().unique())
    industries = sorted(set(sub["industry"].dropna()))
    check_rows.append({
        "entity_id": entity_id, "topic_id": topic_id, "n_iniziative": sub["initiative_id"].nunique(),
        "sentiment_distinti": sentiments, "sentiment_coerente": len(sentiments) <= 1,
        "reason_distinti": reasons, "reason_coerente": len(reasons) <= 1,
        "industry_distinti": industries, "industry_coerente": len(industries) <= 1,
    })

consistency_topic = pd.DataFrame(check_rows)
print(consistency_topic[["sentiment_coerente", "reason_coerente", "industry_coerente"]].sum())
print()
# eventuali incoerenze su QUALUNQUE dei tre campi
consistency_topic[~(
    consistency_topic.sentiment_coerente
    & consistency_topic.reason_coerente
    & consistency_topic.industry_coerente
)]

coppie (entita, topic) con piu' di un'iniziativa distinta: 29
sentiment_coerente    29
reason_coerente       29
industry_coerente     29
dtype: int64



,entity_id,topic_id,n_iniziative,sentiment_distinti,sentiment_coerente,reason_distinti,reason_coerente,industry_distinti,industry_coerente


verifica che (entità, sentiment) sia la stessa in tutti e tre i dataset

In [51]:
G_journ = graphs["journalist"]

rows = []
for u, v, d in G_journ.edges(data=True):
    if d.get("role") != "participant":
        continue
    initiative_id = initiative_of(G_journ, u)
    topic_id = topic_of_node(G_journ, initiative_id) or topic_of_node(G_journ, u)
    rows.append({
        "entity_id": v, "topic_id": topic_id, "initiative_id": initiative_id,
        "sentiment": d.get("sentiment"), "reason": d.get("reason"),
    })
df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["entity_id", "topic_id", "initiative_id", "sentiment", "reason"])

counts = df.groupby(["entity_id", "topic_id"])["initiative_id"].nunique()
multi_pairs = counts[counts > 1].index
print("coppie (entita, topic) con piu' di un'iniziativa distinta:", len(multi_pairs))

check_rows = []
for entity_id, topic_id in multi_pairs:
    sub = df[(df.entity_id == entity_id) & (df.topic_id == topic_id)]
    sentiments = sorted(sub["sentiment"].dropna().unique())
    check_rows.append({
        "entity_id": entity_id, "topic_id": topic_id,
        "sentiment_coerente": len(sentiments) <= 1,
    })
consistency_topic = pd.DataFrame(check_rows)
print(consistency_topic["sentiment_coerente"].value_counts())

coppie (entita, topic) con piu' di un'iniziativa distinta: 29
sentiment_coerente
True    29
Name: count, dtype: int64


## ANALISI PLACES

### LAT E LON SCAMBIATE

In [52]:
# Verifica: road_map.json condivide gli id con i 'place' di journalist? E i campi
# lat/lon sono davvero scambiati rispetto alla convenzione standard?

with open("../public/data/raw_data/road_map.json") as f:
    road_map = json.load(f)
road_map_by_id = {n["id"]: n for n in road_map["nodes"]}

places_journ = {n: d for n, d in G_journ.nodes(data=True) if d.get("type") == "place"}

# 1) sovrapposizione degli id
overlap = set(places_journ) & set(road_map_by_id)
print(f"place in journalist: {len(places_journ)}")
print(f"nodi in road_map.json: {len(road_map_by_id)}")
print(f"sovrapposizione (stesso id): {len(overlap)} "
      f"({len(overlap) / len(places_journ):.0%} dei place di journalist)")
print()

# 2) confronto diretto dei valori numerici, per verificare lo scambio lat/lon
print("--- confronto campi (primi 3 place in comune) ---")
for pid in list(overlap)[:3]:
    j = places_journ[pid]
    r = road_map_by_id[pid]
    print(f"id={pid}")
    print(f"  journalist: lat={j.get('lat')}  lon={j.get('lon')}")
    print(f"  road_map:   latitude={r.get('latitude')}  longitude={r.get('longitude')}")
    print(f"  -> journalist.lat == road_map.longitude ? {j.get('lat') == r.get('longitude')}")
    print(f"  -> journalist.lon == road_map.latitude ? {j.get('lon') == r.get('latitude')}")
print()

# 3) verifica su TUTTI i place in comune (non solo i primi 3)
swap_confirmed = sum(
    1 for pid in overlap
    if places_journ[pid].get("lat") == road_map_by_id[pid].get("longitude")
    and places_journ[pid].get("lon") == road_map_by_id[pid].get("latitude")
)
print(f"place con lat/lon scambiati confermato: {swap_confirmed}/{len(overlap)}")
print()

place in journalist: 173
nodi in road_map.json: 2664
sovrapposizione (stesso id): 173 (100% dei place di journalist)

--- confronto campi (primi 3 place in comune) ---
id=416797704
  journalist: lat=-165.69082493674676  lon=39.00455045258907
  road_map:   latitude=39.00455045258907  longitude=-165.69082493674676
  -> journalist.lat == road_map.longitude ? True
  -> journalist.lon == road_map.latitude ? True
id=582382094
  journalist: lat=-164.57564396402526  lon=39.261334785821575
  road_map:   latitude=39.261334785821575  longitude=-164.57564396402526
  -> journalist.lat == road_map.longitude ? True
  -> journalist.lon == road_map.latitude ? True
id=Suna Spit
  journalist: lat=-165.75677149453168  lon=39.594240090235004
  road_map:   latitude=39.594240090235004  longitude=-165.75677149453168
  -> journalist.lat == road_map.longitude ? True
  -> journalist.lon == road_map.latitude ? True

place con lat/lon scambiati confermato: 172/173



### PLACE NAMES MANCANTI IN JOURNALIST, PRESENTI IN ROADMAP

In [53]:
# 4) i place senza 'name' in journalist: recuperabili da road_map.json?
senza_nome = {n for n, d in places_journ.items() if not d.get("name")}
recuperabili = {n for n in senza_nome if road_map_by_id.get(n, {}).get("name")}
print(f"place senza 'name' in journalist: {len(senza_nome)}")
print(f"di questi, con un 'name' disponibile in road_map.json: {len(recuperabili)}")
print()

# esempi concreti: qualche place senza nome, SOLO i campi comparabili (name vs name)
print("--- esempi di place senza 'name' (prime 10 righe) ---")
esempi = pd.DataFrame([
    {
        "id": n,
        "zone": places_journ[n].get("zone"),
        "zone_detail": places_journ[n].get("zone_detail"),
        "name_in_journalist": places_journ[n].get("name"),       # sempre None/mancante, per costruzione
        "name_in_road_map": road_map_by_id.get(n, {}).get("name"),  # confronto diretto, stesso campo
    }
    for n in list(senza_nome)[:10]
])
esempi

place senza 'name' in journalist: 130
di questi, con un 'name' disponibile in road_map.json: 13

--- esempi di place senza 'name' (prime 10 righe) ---


,id,zone,zone_detail,name_in_journalist,name_in_road_map
0,416797704,commercial,None,None,None
1,3753651731,industrial,None,None,High Seas Fishing Inc.
2,48476691,commercial,None,None,None
3,35908634,tourism,None,None,Harbor's Edge Grill
4,35844638,tourism,None,None,Captain's Market
5,37830690,commercial,None,None,None
6,48462883,commercial,None,None,None
7,12698974244,commercial,None,None,None
8,36952113,commercial,None,None,None
9,36959793,residential,None,None,None


(sopra) dove il nome non c'è ma in read map si, è presente anche in journlaist ma sotto l'attributo 'label'. Verifichiamo (sotto) che label e name siano sempre uguali

In [54]:
# Verifica: quando 'name' e 'label' sono entrambi presenti (non None) su un place,
# hanno sempre lo stesso valore? E quanti place hanno 'label' ma non 'name'
# (il caso che hai appena scoperto, potenzialmente recuperabile)?

place_nodes = {n: d for n, d in G_journ.nodes(data=True) if d.get("type") == "place"}

both_present = {n: d for n, d in place_nodes.items() if d.get("name") and d.get("label")}
mismatches = {n: d for n, d in both_present.items() if d.get("name") != d.get("label")}

print(f"place con sia 'name' sia 'label' presenti: {len(both_present)}")
print(f"di questi, con valori DIVERSI (mismatch): {len(mismatches)}")
if mismatches:
    for n, d in mismatches.items():
        print(f"  {n}: name={d.get('name')!r}  label={d.get('label')!r}")
print()

# il caso che hai trovato: 'name' assente ma 'label' presente -> recuperabile da journalist stesso
name_assente_label_presente = {
    n: d for n, d in place_nodes.items() if not d.get("name") and d.get("label")
}
print(f"place SENZA 'name' ma CON 'label': {len(name_assente_label_presente)}")
for n, d in list(name_assente_label_presente.items())[:10]:
    print(f"  {n}: label={d.get('label')!r}")

place con sia 'name' sia 'label' presenti: 6
di questi, con valori DIVERSI (mismatch): 0

place SENZA 'name' ma CON 'label': 13
  35889363: label='Harbor Route Solutions'
  3753651731: label='High Seas Fishing Inc.'
  491334376: label='Blue Wave Shipping'
  30192735: label='Bay Harvest Corporation'
  35922181: label='Harbor Builders'
  35872967: label='Haacklee Assembly Co.'
  48468290: label='Waveside Townhomes'
  48480127: label='Tidewater Flats'
  5849775342: label="Sailor's Perch Light"
  35844638: label="Captain's Market"


## IN TROUT CI SONO TUTTE LE PERSONE, ALCUNE SENZA ARCHI COLLEGATI

In [55]:
def person_present_but_silent(graphs, person_type="entity.person"):
    """Per ogni persona presente come NODO in un dataset di parte, verifica se ha
    ALMENO UN arco 'participant' in quel dataset, oppure se e' presente ma "muta"
    (nodo noto, ma nessuna opinione/partecipazione mai registrata)."""
    G_journ = graphs["journalist"]
    persons = {n for n, d in G_journ.nodes(data=True) if d.get("type") == person_type}

    rows = []
    for name in ["FILAH", "TROUT"]:
        G_sub = graphs[name]
        for person in persons:
            nodo_presente = person in G_sub.nodes
            if not nodo_presente:
                continue  # gia' sappiamo che manca del tutto, non e' il caso che ci interessa qui
            n_participant = sum(
                1 for _, v, d in G_sub.edges(data=True)
                if d.get("role") == "participant" and v == person
            )
            rows.append({
                "dataset": name, "person": person,
                "nodo_presente": nodo_presente,
                "n_archi_participant": n_participant,
                "presente_ma_muta": n_participant == 0,
            })
    return pd.DataFrame(rows)


silent_check = person_present_but_silent(graphs)
print(silent_check.groupby("dataset")["presente_ma_muta"].value_counts())
silent_check[silent_check["presente_ma_muta"]]

dataset  presente_ma_muta
FILAH    False               3
TROUT    False               3
         True                3
Name: count, dtype: int64


,dataset,person,nodo_presente,n_archi_participant,presente_ma_muta
3,TROUT,Carol Limpet,True,0,True
7,TROUT,Simone Kat,True,0,True
8,TROUT,Tante Titan,True,0,True
